In [ ]:
import re
import json
import tables
import joblib
import logging
import numpy as np
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from scipy.stats import pearsonr
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import torch
from torch import nn, optim

if torch.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
dtype = torch.float32

In [ ]:
import sys
if '..' not in sys.path:
    sys.path = ['..'] + sys.path
from prepare_data_cnn_regressor import make_dataloaders
from cnn_regressor import DeepFilteringRegressor
from train_cnn_regressor import train_model, load_checkpoint, MeanAbsoluteRelativeError

In [ ]:
def check_dir(D):
    return (
        re.match('^[0-9]+$', D.name) is not None and
        any(re.match(data_file_pattern, f.name) for f in D.glob('*.h5'))
    )

def load_h5_file(fname, dtype=np.float32):
    fid = tables.open_file(fname)
    parameters = fid.root.parameters.read()
    time = fid.root.time.read()
    dt = time[1] - time[0]
    var_names, X = [], []
    for child in fid.root:
        if child.name not in ('time', 'parameters'):
            var_names.append(child.name)
            x = child.read()
            # check shape of x to make sure that we flatten appropriately
            order = 'C' if x.shape[0] < x.shape[1] else 'F'
            X.append(x.flatten(order))
    fid.close()
    idx = [var_names.index(_) for _ in sorted(var_names)]
    var_names = [var_names[i] for i in idx]
    X = np.vstack([X[i] for i in idx]).astype(dtype)
    time = (np.arange(X[0].size) * dt).astype(dtype)
    return time, X, var_names, parameters

def load_measure(fname, meas_name='M'):
    with open(fname) as fid:
        D = json.load(fid)
    try:
        return D[meas_name]
    except:
        long_to_short = {'inertia': 'H', 'energy': 'E', 'momentum': 'M'}
        key = meas_name.lower()
        if key in long_to_short:
            return D[long_to_short[key]]
        raise KeyError(f"Unknown measure name '{meas_name}'")

In [ ]:
_NAMES = {'H': 'inertia', 'E': 'energy', 'M': 'momentum'}
_UNITS = {'H': 's', 'E': 'MWs', 'M': 'MWs2'}
_LATEX_UNITS = {'H': '$s$', 'E': '$MW\\cdot s$', 'M': '$MW\\cdot s^2$'}
_COEFFS = {'H': 1, 'E': 1e-3, 'M': 1e-3}

In [ ]:
experiments_dir = Path('experiments')
if not experiments_dir.exists():
    experiments_dir.mkdir(mode=0o755, exist_ok=True)

suffix_len = 8
suffix = ''
a, z = ord('a'), ord('z')
zero, nine = ord('0'), ord('9')
with open('/dev/random', 'rb') as fid:
    while True:
        try:
            b = ord(fid.read(1).decode('ascii'))
            if (b >= a and b <= z) or (b >= zero and b <= nine):
                suffix += chr(b)
                if len(suffix) == suffix_len:
                    break
        except:
            pass
expt_dir = experiments_dir / suffix
expt_dir.mkdir(mode=0o755, exist_ok=False)
print(f'Will save experiment results to {expt_dir}.')

checkpoint_path = expt_dir / 'best_model.pt'
logger_path = expt_dir / 'output.log'
logger = logging.getLogger()
logging.basicConfig(filename=logger_path, level=logging.INFO)
logger.addHandler(logging.StreamHandler(sys.stdout))

In [ ]:
base_data_dir = Path('../data/Sardinia/SM_configs_from_data/stoch_loads_no_GFM')
data_file_pattern = 'Sardegna_stoch_loads_no_GFM_TF_-6\\.0_2\\.0_50_training_set\\.h5'
data_dirs = sorted(filter(check_dir, base_data_dir.glob('*')))
n_data_dirs = len(data_dirs)
logger.info(f'Found {n_data_dirs} directories.')

In [ ]:
measure_name = 'M'
X, y, var_names = [], [], []
for D in data_dirs:
    data_files = filter(lambda f: re.match(data_file_pattern, f.name) is not None, D.glob('*.h5'))
    meas = load_measure(D / 'HEM.json', measure_name)
    for fname in data_files:
        time, x, names, _ = load_h5_file(fname)
        X.append(x)
        y.append(meas)
        var_names.append(names)
y = np.array(y) * _COEFFS[measure_name]
dt = time[1] - time[0]
assert all(var_names[0] == names for names in var_names)
var_names = var_names[0]
n_vars = len(var_names)

#### Parameters

In [ ]:
# data parameters
trial_dur = 60   # [s]
batch_size = 64
scale_target = True
# optimizer parameters
lr = 2e-3
betas = [0.9, 0.999]
eps = 1e-8
weight_decay = 1e-3
# training parameters
n_epochs = 100
patience = 20
scheduler_name = 'exponential'

In [ ]:
n_samples_per_trial = int(np.ceil(trial_dur / dt)) + 1
n_trials = X[0].shape[1] // n_samples_per_trial
X = np.concat(
    [x[:, :n_trials * n_samples_per_trial].reshape(n_vars, -1, n_samples_per_trial) for x in X],
    axis=1,
).transpose((1, 0, 2))
y = np.tile(y.reshape(-1, 1), (n_trials,)).flatten().reshape(-1, 1)
N, C, L = X.shape
logger.info(f'Number of trials: {N}')
logger.info(f'Number of channels: {C}')
logger.info(f'Sequence length: {L}')

In [ ]:
data_loaders, scalers = make_dataloaders(
    X,
    y,
    val_size=0.15,
    test_size=0.15,
    batch_size=batch_size,
    scale_target=scale_target,
    device=device,
    dtype=dtype,
)
joblib.dump(scalers['x'], expt_dir / 'x_scaler.joblib')
if scale_target:
    joblib.dump(scalers['y'], expt_dir / 'y_scaler.joblib')

In [ ]:
config = dict(
    num_channels=C,
    input_length=L,
    num_layers=2,
    kernel_size=[16, 8],
    pool_size=4,
    filters=[8, 16],
    hidden_units=[32],
    device=device,
    dtype=dtype,
)
model = DeepFilteringRegressor(**config)

In [ ]:
pars = [(_[0], _[1].numel()) for _ in model.named_parameters()]
strlen = max(len(_[0]) for _ in pars)
logger.info(model)
logger.info('WEIGHTS')
logger.info('=======')
total = 0
for i, (name, num) in enumerate(pars):
    total += num
    logger.info('[{:2d}] {} {} {:,}'.format(i + 1, name, '.' * (5 + strlen - len(name)), num))
logger.info(f'Total number of parameters: {total:,}.')

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr, betas, eps, weight_decay)

In [ ]:
if scheduler_name is not None:
    if scheduler_name == 'plateau':
        scheduler_ctor = optim.lr_scheduler.ReduceLROnPlateau
        scheduler_kwargs = {'mode': 'min', 'factor': 0.5, 'patience': 10}
    elif scheduler_name == 'exponential':
        scheduler_ctor = optim.lr_scheduler.ExponentialLR
        lr_final = lr / 20
        gamma = (lr_final / lr) ** (1 / n_epochs)
        scheduler_kwargs = {'gamma': gamma}
        logger.info(f'Decay parameter of exponential learning rate: {gamma:g}')
    else:
        raise Exception(f"Unknown scheduler '{scheduler_name}'")
    scheduler = scheduler_ctor(optimizer, **scheduler_kwargs)
else:
    scheduler = None

In [ ]:
if not scale_target:
    loss_fun = MeanAbsoluteRelativeError()
else:
    loss_fun = nn.MSELoss()
    loss_fun = nn.L1Loss()
result = train_model(
    model,
    data_loaders['train'],
    data_loaders['val'],
    checkpoint_path=checkpoint_path,
    loss_fn=loss_fun,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    patience=patience,
    model_config=config,
    device=device,
    logger=logger,
)

In [ ]:
history = result['history']
epoch = np.array([_['epoch'] for _ in history])
val_loss = np.array([_['val_loss'] for _ in history])
train_loss = np.array([_['train_loss'] for _ in history])
learning_rate = np.array([_['lr'] for _ in history])

fig, ax = plt.subplots(2, 1, figsize=(4, 4), sharex=True, layout='constrained')
ax[0].plot(epoch[1:], train_loss[1:], 'k', lw=1, label='Train')
ax[0].plot(epoch, val_loss, 'tab:red', lw=1, label='Validation')
ax[0].set_ylabel('Loss')
ax[0].legend(loc='upper right', frameon=False, fontsize=8)
ax[0].set_yscale('log')
ax[1].plot(epoch, learning_rate, 'k', lw=1)
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Learning rate')
sns.despine()
plt.savefig(expt_dir / 'history.pdf')

Reload the best model, i.e., the one with the lowest validation loss. This call here is not strictly necessary, since `train_model` loads the state dictionary of the best checkpoint at the end of the training procedure.

In [ ]:
best_model, ckpt = load_checkpoint(checkpoint_path, device=device)

In [ ]:
tensor_to_numpy = lambda t: t.cpu().numpy().squeeze()
L = []
y_test, y_test_pred = [], []
with torch.no_grad():
    for (x, y_true) in data_loaders['test']:
        y_pred = best_model(x)
        L.append(loss_fun(y_true, y_pred).item())
        y_test.append(tensor_to_numpy(y_true))
        y_test_pred.append(tensor_to_numpy(y_pred))
loss = np.mean(L)
logger.info(f'Test set loss: {loss:g}')

In [ ]:
y_test = np.concat(y_test).reshape(-1, 1)
y_test_pred = np.concat(y_test_pred).reshape(-1, 1)
if 'y' in scalers:
    y_test = scalers['y'].inverse_transform(y_test)
    y_test_pred = scalers['y'].inverse_transform(y_test_pred)
y_test =  y_test.squeeze() / _COEFFS[measure_name]
y_test_pred =  y_test_pred.squeeze() / _COEFFS[measure_name]

In [ ]:
RMSE = root_mean_squared_error(y_test, y_test_pred)
MAE = mean_absolute_error(y_test, y_test_pred)
MAPE = mean_absolute_percentage_error(y_test, y_test_pred)
R2 = pearsonr(y_test, y_test_pred).statistic
logger.info('RMSE ... {:6.3f} {}'.format(RMSE, _UNITS[measure_name]))
logger.info('MAE .... {:6.3f} {}'.format(MAE, _UNITS[measure_name]))
logger.info('MAPE ... {:6.3f}%'.format(MAPE * 100))
logger.info('R2 ..... {:6.3f}'.format(R2, _UNITS[measure_name]))

In [ ]:
lim = np.array([min(y_test.min(), y_test_pred.min()), min(y_test.max(), y_test_pred.max())])
dy = lim[1] - lim[0]
lim += np.array([-1, 1]) * dy * 0.05
fig, ax = plt.subplots(1, 1, figsize=(3, 2.5), layout='constrained')
ax.plot(lim, lim, color='tab:red', lw=2)
ax.scatter(y_test, y_test_pred, c='k', s=5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel(r'{} [{}]'.format(_NAMES[measure_name].capitalize(), _LATEX_UNITS[measure_name]))
ax.set_ylabel(r'Predicted {} [{}]'.format(_NAMES[measure_name], _LATEX_UNITS[measure_name]))
ax.grid(which='major', axis='both', ls=':', lw=0.5, color=[.6,.6,.6])
sns.despine()
plt.savefig(expt_dir / 'predictions.pdf')